# Day 42 / 42: MLOps Basics

**42 Days of ML Challenge**

Yesterday you deployed a model behind an API. Today's question: how do you know if that model is still good six months from now?

Nobody manually re-checks a deployed model every day. MLOps is the set of practices that answers three questions automatically: which experiment produced the model you shipped, which version is actually running, and has the world changed enough that the model needs retraining.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week6_production/day42_mlops_basics/day42_notebook.ipynb)

---

### What You'll Learn
- Experiment tracking with MLflow: logging params, metrics, and comparing runs
- Model versioning: how to know exactly which model is in production
- Data drift detection: a statistical way to catch when incoming data no longer looks like training data
- Why "the model still runs" is not the same as "the model still works"

---


## Setup

If you're in Google Colab, run this cell first. If you're in Jupyter Lab/Notebook locally, make sure these packages are installed in your environment.

In [1]:
!pip install mlflow scikit-learn pandas scipy -q

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

---
## Part 1: The Problem MLOps Actually Solves

By Day 42, you've trained dozens of models across this series. In a notebook, that's fine, you can scroll up and see what you ran. In a real team, that breaks down fast:

- Which of the 12 experiments last week actually produced the model in production?
- What hyperparameters did it use? What was the exact training data?
- Six months later, is the model's accuracy quietly dropping because customer behavior changed?

None of these are modelling questions. They're **operational** questions. That's what MLOps is: the practices and tooling that make ML reproducible, trackable, and monitorable after the notebook closes.

Today covers the three basics every MLE is expected to know: **experiment tracking**, **model versioning**, and **drift monitoring**.

---
## Part 2: Experiment Tracking with MLflow

Reusing the Titanic dataset one more time, but the point today isn't the model, it's tracking *how* you got there. We'll train 3 model variants and log every parameter and metric automatically instead of writing them down by hand or, worse, not writing them down at all.

In [2]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

FEATURES = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]
df = df[FEATURES + ["Survived"]].copy()
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

X = df[FEATURES]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data ready:", X_train.shape, X_test.shape)

Data ready: (712, 6) (179, 6)


### Set up MLflow tracking

MLflow needs a place to store experiment logs. We're using a local SQLite database, which works identically in Jupyter and Colab, no external server required. In a real team, this tracking URI would point to a shared MLflow server so everyone sees the same experiment history.

In [3]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("titanic_survival")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Active experiment:", mlflow.get_experiment_by_name("titanic_survival").name)

2026/07/11 06:26:35 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/07/11 06:26:35 INFO mlflow.store.db.utils: Updating database tables


2026/07/11 06:26:37 INFO mlflow.tracking.fluent: Experiment with name 'titanic_survival' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db
Active experiment: titanic_survival


### Train 3 model variants and log each run

Instead of manually noting down accuracy in a comment or a separate spreadsheet, `mlflow.start_run()` captures the model type, its parameters, and its metrics automatically, tied to a unique run ID.

In [4]:
models_to_try = {
    "logistic_regression": LogisticRegression(max_iter=1000),
    "random_forest_n50": RandomForestClassifier(n_estimators=50, random_state=42),
    "random_forest_n200": RandomForestClassifier(n_estimators=200, random_state=42),
}

for name, model in models_to_try.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)

        # Log what was used
        mlflow.log_param("model_type", name)
        if hasattr(model, "n_estimators"):
            mlflow.log_param("n_estimators", model.n_estimators)

        # Log how it performed
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)

        print(f"{name:22s} accuracy={acc:.4f}  f1={f1:.4f}")

logistic_regression    accuracy=0.8045  f1=0.7368


random_forest_n50      accuracy=0.8212  f1=0.7576


random_forest_n200     accuracy=0.8212  f1=0.7576


### Compare runs and pick the best one

This is the actual payoff: instead of scrolling back through notebook cells to remember which run performed best, you query the tracking store directly.

In [5]:
runs = mlflow.search_runs(experiment_names=["titanic_survival"])
comparison = runs[["tags.mlflow.runName", "metrics.accuracy", "metrics.f1_score"]].sort_values(
    "metrics.f1_score", ascending=False
)
comparison.columns = ["run_name", "accuracy", "f1_score"]
print(comparison.to_string(index=False))

best_run = comparison.iloc[0]
print(f"\nBest run by F1: {best_run['run_name']} (f1={best_run['f1_score']:.4f})")

           run_name  accuracy  f1_score
 random_forest_n200  0.821229  0.757576
  random_forest_n50  0.821229  0.757576
logistic_regression  0.804469  0.736842

Best run by F1: random_forest_n200 (f1=0.7576)


**Real world:** DoorDash's ML team uses MLflow to track 200+ experiments before selecting a delivery ETA model for production. The value isn't the tool itself, it's that six months later, anyone on the team can answer "why did we pick this model?" with an actual logged answer instead of a guess.

---
## Part 3: Model Versioning

Training runs tell you what was *tried*. Versioning tells you what's actually *deployed*. These are different things, a team might run 50 experiments and ship exactly one.

The simplest reliable pattern: save every deployed model with a version tag and the metrics it shipped with, so the API from Day 41 always knows exactly which model it's serving.

In [6]:
import joblib
from datetime import datetime

# Retrain the winning model type on the full pipeline for deployment
final_model = RandomForestClassifier(n_estimators=200, random_state=42)
final_model.fit(X_train_scaled, y_train)

version_tag = datetime.now().strftime("v%Y%m%d_%H%M%S")

model_registry = {
    "version": version_tag,
    "model_type": "random_forest_n200",
    "features": FEATURES,
    "test_accuracy": round(accuracy_score(y_test, final_model.predict(X_test_scaled)), 4),
    "test_f1": round(f1_score(y_test, final_model.predict(X_test_scaled)), 4),
    "trained_on_rows": len(X_train),
}

joblib.dump(final_model, f"model_{version_tag}.joblib")
joblib.dump(scaler, f"scaler_{version_tag}.joblib")

import json
with open(f"model_{version_tag}_metadata.json", "w") as f:
    json.dump(model_registry, f, indent=2)

print("Deployed model version:", version_tag)
print(json.dumps(model_registry, indent=2))

Deployed model version: v20260711_062638
{
  "version": "v20260711_062638",
  "model_type": "random_forest_n200",
  "features": [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
  ],
  "test_accuracy": 0.8212,
  "test_f1": 0.7576,
  "trained_on_rows": 712
}


**Why this matters:** if a model in production starts behaving strangely, the first question in any incident is "which version is actually running, and what was it trained on?" A model file with no version tag and no metadata attached to it cannot answer that question. This is a 10-line habit that saves hours of production debugging.

---
## Part 4: Data Drift Detection

A model doesn't get worse on its own, the world around it changes. Customer behavior shifts, a competitor launches, a pricing change happens, and the data the model sees in production slowly stops resembling the data it was trained on. This is called **data drift**, and it's the single most common reason a deployed model quietly degrades.

We'll simulate this: take the training data, then simulate 3 months of "new" incoming data where the `Age` distribution has shifted (imagine a marketing campaign that suddenly attracted an older customer base). Then detect it statistically instead of guessing.

In [7]:
from scipy import stats

# Reference: the distribution the model was trained on
reference_data = X_train.copy()

# Simulate new incoming production data with a shifted Age distribution
np.random.seed(7)
incoming_data = X_test.copy()
incoming_data["Age"] = incoming_data["Age"] + np.random.normal(15, 5, len(incoming_data))

print("Reference Age  -> mean:", round(reference_data["Age"].mean(), 1), " std:", round(reference_data["Age"].std(), 1))
print("Incoming Age   -> mean:", round(incoming_data["Age"].mean(), 1), " std:", round(incoming_data["Age"].std(), 1))

Reference Age  -> mean: 29.5  std: 13.0
Incoming Age   -> mean: 43.8  std: 14.4


### The Kolmogorov-Smirnov test

The KS test checks whether two samples come from the same distribution. It returns a p-value: if it's below your threshold (typically 0.05), the distributions are statistically different enough to flag as drift.

In [8]:
def detect_drift(reference, current, column, alpha=0.05):
    stat, p_value = stats.ks_2samp(reference[column], current[column])
    return {
        "column": column,
        "ks_statistic": round(stat, 4),
        "p_value": round(p_value, 4),
        "drift_detected": bool(p_value < alpha),
    }

print("Drift report: reference training data vs incoming production data\n")
for col in ["Age", "Fare", "Pclass", "SibSp"]:
    result = detect_drift(reference_data, incoming_data, col)
    flag = "DRIFT DETECTED" if result["drift_detected"] else "no drift"
    print(f"  {col:8s} | ks={result['ks_statistic']:.4f} | p={result['p_value']:.4f} | {flag}")

Drift report: reference training data vs incoming production data

  Age      | ks=0.5332 | p=0.0000 | DRIFT DETECTED
  Fare     | ks=0.0523 | p=0.8053 | no drift
  Pclass   | ks=0.0112 | p=1.0000 | no drift
  SibSp    | ks=0.0569 | p=0.7157 | no drift


Notice `Age` gets flagged and the others don't, exactly as expected, since we only shifted `Age`. In production, this check would run automatically on every new batch of incoming data, and a flagged column would trigger an alert to the ML team rather than being discovered 3 months later when someone finally asks why accuracy looks off.

**Real world:** this is the same category of check LinkedIn's feed ranking team runs continuously in production, comparing live feature distributions against training-time distributions to catch silent model decay before it shows up in business metrics.

---
## Real World Problem

An e-commerce company deploys a churn prediction model. It performs well for the first 4 months. In month 5, a major pricing change goes live. Customer behavior shifts overnight, but the model keeps running unchanged, still making predictions based on pre-pricing-change patterns.

Nobody notices for 6 weeks, because the model doesn't crash. It just gets quietly worse. Business stakeholders eventually notice churn interventions aren't working, and by the time anyone investigates, the team has lost weeks of effective targeting.

The fix isn't a better model. It's the drift check from this notebook running automatically on every batch of incoming data, with an alert wired up before the business impact, not after.

---
## Interview Corner

**Q: How do you know if a model deployed in production is still performing well?**

**What they're testing:** Whether you think about ML as a one-time deployment or an ongoing operational responsibility.

**Answer direction:**
- A model doesn't announce its own failure, you have to monitor for it
- Two signals matter: data drift (are incoming features still shaped like training data) and prediction drift (has the distribution of the model's own outputs shifted unexpectedly)
- Statistical tests like KS-test or PSI (Population Stability Index) can flag drift automatically on scheduled batches of incoming data
- Where possible, tie monitoring back to a ground-truth business metric, not just statistical drift, since some drift is harmless and some isn't
- Mention that experiment tracking and model versioning are what make it possible to quickly retrain and redeploy once drift is confirmed

---
## ML Spotlight

**MLflow** is one of the most widely used open source tools for experiment tracking and model registry in production ML teams. Beyond what we used today, it also supports a full **Model Registry** for staging/production promotion workflows, meaning a model can move through "staging" to "production" with an audit trail of who approved it and when.

Docs: [mlflow.org/docs/latest](https://mlflow.org/docs/latest)

---
## Key Takeaways

- Experiment tracking answers "why did we pick this model", automatically, not from memory
- Model versioning means every deployed model can answer "what am I, and what was I trained on"
- Data drift is the most common reason a model quietly degrades after deployment
- Statistical drift tests (KS-test, PSI) can be automated and run on every new batch of production data
- MLOps isn't a separate skill from ML, it's what makes ML durable past the first deployment

---

### What's Next

**Day 43: The Capstone.** Every phase of this series comes together into one project: raw data to a cleaned dataset, feature engineering, model selection, a deployed FastAPI endpoint, and drift monitoring wired in. Built to be resume-ready, not just a notebook.

Full code: [github.com/VaishnaviJagtap18/42-days-aiml-challenge](https://github.com/VaishnaviJagtap18/42-days-aiml-challenge)

#42DaysOfML
